# Voice Deepfake Detector: XLS-R 300M + AASIST-style back-end + RawBoost

**Before you run (read once):**
1. Runtime > Change runtime type > **GPU** (T4 is fine).
2. Upload `colab_pack.zip` to the **root of your Google Drive** (My Drive).
3. Run cells top to bottom. This notebook was **not tested on a GPU before you got it**, so if a cell errors, paste the error back.

**Honest notes**
- The back-end is a **simplified AASIST-style re-implementation** (residual encoder + spectral/temporal graph attention + master nodes). It is **not the official AASIST code**. Say "AASIST-style" on the resume.
- Train = LA train subset (4000). Dev = LA dev (used only to pick the best epoch). Eval and MLAAD are **never** used for training. RawBoost is applied to training data only.
- Everything is a subset and a single run (one seed). Do not claim small differences as real.
- MLAAD real/fake come from different sources (noise-floor leak, found earlier). So also look at **mlaad_matched** (noise floor matched, only 225 pairs).
- Colab free GPU is not guaranteed: it can disconnect or run out of quota. The best checkpoint is saved to Drive after every improvement.

**Earlier results on your Mac (baseline CNN, for comparison):**
| model | LA eval EER | MLAAD all EER | MLAAD matched EER |
|---|---|---|---|
| CNN | 5.30% | 63.0% | 56.2% |
| CNN + RawBoost algo 4 | 7.00% | 34.1% | 45.8% |


In [ ]:
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.is_available() and torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Turn on GPU: Runtime > Change runtime type"
!pip -q install transformers soundfile scikit-learn scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ZIP = "/content/drive/MyDrive/colab_pack.zip"
!rm -rf /content/colab_pack && mkdir -p /content/colab_pack
!cp "$ZIP" /content/colab_pack.zip && unzip -q /content/colab_pack.zip -d /content/colab_pack
%cd /content/colab_pack
!ls; ls data; ls audio/train | head -3; ls audio/train | wc -l

In [ ]:
%%writefile RawBoost.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import numpy as np
from scipy import signal
import copy

'''
   Hemlata Tak, Madhu Kamble, Jose Patino, Massimiliano Todisco, Nicholas Evans.
   RawBoost: A Raw Data Boosting and Augmentation Method applied to Automatic Speaker Verification Anti-Spoofing.
   In Proc. ICASSP 2022, pp:6382--6386.
'''

def randRange(x1, x2, integer):
    # PATCHED for new NumPy: return plain python numbers (original returned size-1 arrays)
    y = float(np.random.uniform(low=x1, high=x2))
    if integer:
        y = int(y)
    return y

def normWav(x,always):
    if always:
        x = x/np.amax(abs(x))
    elif np.amax(abs(x)) > 1:
            x = x/np.amax(abs(x))
    return x


def genNotchCoeffs(nBands,minF,maxF,minBW,maxBW,minCoeff,maxCoeff,minG,maxG,fs):
    b = 1
    for i in range(0, nBands):
        fc = randRange(minF,maxF,0);
        bw = randRange(minBW,maxBW,0);
        c = randRange(minCoeff,maxCoeff,1);
          
        if c/2 == int(c/2):
            c = c + 1
        f1 = fc - bw/2
        f2 = fc + bw/2
        if f1 <= 0:
            f1 = 1/1000
        if f2 >= fs/2:
            f2 =  fs/2-1/1000
        b = np.convolve(signal.firwin(c, [float(f1), float(f2)], window='hamming', fs=fs),b)

    G = randRange(minG,maxG,0); 
    _, h = signal.freqz(b, 1, fs=fs)    
    b = pow(10, G/20)*b/np.amax(abs(h))   
    return b


def filterFIR(x,b):
    N = b.shape[0] + 1
    xpad = np.pad(x, (0, N), 'constant')
    y = signal.lfilter(b, 1, xpad)
    y = y[int(N/2):int(y.shape[0]-N/2)]
    return y

# Linear and non-linear convolutive noise
def LnL_convolutive_noise(x,N_f,nBands,minF,maxF,minBW,maxBW,minCoeff,maxCoeff,minG,maxG,minBiasLinNonLin,maxBiasLinNonLin,fs):
    y = [0] * x.shape[0]
    for i in range(0, N_f):
        if i == 1:
            minG = minG-minBiasLinNonLin;
            maxG = maxG-maxBiasLinNonLin;
        b = genNotchCoeffs(nBands,minF,maxF,minBW,maxBW,minCoeff,maxCoeff,minG,maxG,fs)
        y = y + filterFIR(np.power(x, (i+1)),  b)     
    y = y - np.mean(y)
    y = normWav(y,0)
    return y


# Impulsive signal dependent noise
def ISD_additive_noise(x, P, g_sd):
    beta = randRange(0, P, 0)
    
    y = copy.deepcopy(x)
    x_len = x.shape[0]
    n = int(x_len*(beta/100))
    p = np.random.permutation(x_len)[:n]
    f_r= np.multiply(((2*np.random.rand(p.shape[0]))-1),((2*np.random.rand(p.shape[0]))-1))
    r = g_sd * x[p] * f_r
    y[p] = x[p] + r
    y = normWav(y,0)
    return y


# Stationary signal independent noise

def SSI_additive_noise(x,SNRmin,SNRmax,nBands,minF,maxF,minBW,maxBW,minCoeff,maxCoeff,minG,maxG,fs):
    noise = np.random.normal(0, 1, x.shape[0])
    b = genNotchCoeffs(nBands,minF,maxF,minBW,maxBW,minCoeff,maxCoeff,minG,maxG,fs)
    noise = filterFIR(noise, b)
    noise = normWav(noise,1)
    SNR = randRange(SNRmin, SNRmax, 0)
    noise = noise / np.linalg.norm(noise,2) * np.linalg.norm(x,2) / 10.0**(0.05 * SNR)
    x = x + noise
    return x



In [ ]:
%%writefile rawboost_aug.py
"""
Wrapper around the official RawBoost.py (Tak et al., ICASSP 2022,
github.com/TakHemlata/SSL_Anti-spoofing). Parameters = official defaults.
algo: 0 none | 1 LnL | 2 ISD | 3 SSI | 4 (1+2+3) | 5 (1+2) | 6 (1+3) | 7 (2+3) | 8 parallel(1,2)
"""
import numpy as np
from RawBoost import LnL_convolutive_noise, ISD_additive_noise, SSI_additive_noise, normWav

P = dict(nBands=5, minF=20, maxF=8000, minBW=100, maxBW=1000, minCoeff=10, maxCoeff=100,
         minG=0, maxG=0, minBias=5, maxBias=20, N_f=5, P=10, g_sd=2, SNRmin=10, SNRmax=40)
FS = 16000

def _lnl(x):
    return LnL_convolutive_noise(x, P["N_f"], P["nBands"], P["minF"], P["maxF"], P["minBW"], P["maxBW"],
                                 P["minCoeff"], P["maxCoeff"], P["minG"], P["maxG"], P["minBias"], P["maxBias"], FS)

def _isd(x):
    return ISD_additive_noise(x, P["P"], P["g_sd"])

def _ssi(x):
    return SSI_additive_noise(x, P["SNRmin"], P["SNRmax"], P["nBands"], P["minF"], P["maxF"], P["minBW"],
                              P["maxBW"], P["minCoeff"], P["maxCoeff"], P["minG"], P["maxG"], FS)

def rawboost(x, algo):
    x = np.asarray(x, dtype=np.float64)
    if algo == 0:
        return x.astype(np.float32)
    if algo == 1:   y = _lnl(x)
    elif algo == 2: y = _isd(x)
    elif algo == 3: y = _ssi(x)
    elif algo == 4: y = _ssi(_isd(_lnl(x)))
    elif algo == 5: y = _isd(_lnl(x))
    elif algo == 6: y = _ssi(_lnl(x))
    elif algo == 7: y = _ssi(_isd(x))
    elif algo == 8: y = normWav(_lnl(x) + _isd(x), 0)
    else: raise ValueError("algo must be 0..8")
    y = np.nan_to_num(np.asarray(y, dtype=np.float64).reshape(-1))
    return y.astype(np.float32)


In [ ]:
%%writefile metrics.py
"""EER, AUC, F1. label 1 = fake (positive class), score = probability of fake."""
import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score, f1_score

def compute_metrics(labels, scores, threshold=0.5):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores, dtype=float)
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    eer = (fpr[i] + fnr[i]) / 2
    return {
        "EER": float(eer * 100),                     # in %, lower is better
        "AUC": float(roc_auc_score(labels, scores)), # closer to 1 is better
        "F1": float(f1_score(labels, (scores >= threshold).astype(int))),
    }


In [ ]:
%%writefile xlsr_aasist.py
"""XLS-R 300M front-end + AASIST-style graph-attention back-end (simplified re-implementation)."""
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

SR = 16000
MAX_LEN = 4 * SR


# ---------------- data ----------------
def fix_length(x, n=MAX_LEN):
    if len(x) >= n:
        return x[:n]
    return np.tile(x, int(np.ceil(n / len(x))))[:n]


class AudioDataset(Dataset):
    """algo>0 -> RawBoost (use ONLY for training data). prob = fraction of clips augmented."""
    def __init__(self, csv_path, algo=0, prob=0.5):
        self.df = pd.read_csv(csv_path)
        self.algo, self.prob = algo, prob
        if algo > 0:
            from rawboost_aug import rawboost
            self._rb = rawboost

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        wav, sr = sf.read(row["path"], dtype="float32")
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        assert sr == SR, f"{row['path']}: sample rate {sr}"
        wav = fix_length(wav)
        if self.algo > 0 and np.random.rand() < self.prob:
            wav = self._rb(wav, self.algo)
        return torch.from_numpy(np.ascontiguousarray(wav)), torch.tensor(row["label"], dtype=torch.float32)


def worker_init(_):
    # every DataLoader worker needs its own numpy seed, otherwise RawBoost noise repeats
    np.random.seed(torch.initial_seed() % (2 ** 32))


# ---------------- back-end ----------------
class ResBlock(nn.Module):
    def __init__(self, cin, cout, pool_t=1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(cin)
        self.c1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.short = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
        self.pool = nn.MaxPool2d((1, pool_t)) if pool_t > 1 else nn.Identity()

    def forward(self, x):
        y = self.c1(F.selu(self.bn1(x)))
        y = self.c2(F.selu(self.bn2(y)))
        return self.pool(y + self.short(x))


class GraphAttn(nn.Module):
    """Attention over a fully connected graph. nodes (B, N, din) -> (B, N, dout)."""
    def __init__(self, din, dout):
        super().__init__()
        self.q, self.k, self.v = nn.Linear(din, dout), nn.Linear(din, dout), nn.Linear(din, dout)
        self.res = nn.Linear(din, dout)
        self.bn = nn.BatchNorm1d(dout)
        self.scale = dout ** -0.5

    def forward(self, h):
        att = torch.softmax(self.q(h) @ self.k(h).transpose(1, 2) * self.scale, dim=-1)
        out = att @ self.v(h) + self.res(h)
        return F.selu(self.bn(out.transpose(1, 2)).transpose(1, 2))


class Branch(nn.Module):
    """Graph attention on one axis (spectral or temporal) + a learnable master node."""
    def __init__(self, d):
        super().__init__()
        self.g1, self.g2 = GraphAttn(d, d), GraphAttn(d, d)
        self.master = nn.Parameter(torch.randn(1, 1, d) * 0.1)

    def forward(self, nodes):
        h = self.g1(nodes)
        hm = self.g2(torch.cat([self.master.expand(h.size(0), -1, -1), h], dim=1))
        return hm[:, 0], hm[:, 1:]


class XLSR_AASIST(nn.Module):
    def __init__(self, ssl_name="facebook/wav2vec2-xls-r-300m", ssl=None, d=64):
        super().__init__()
        if ssl is None:
            from transformers import Wav2Vec2Model
            ssl = Wav2Vec2Model.from_pretrained(ssl_name, layerdrop=0.0, apply_spec_augment=False)
        self.ssl = ssl
        self.ssl.freeze_feature_encoder()            # keep the small CNN part frozen
        self.proj = nn.Linear(self.ssl.config.hidden_size, 128)
        self.bn0 = nn.BatchNorm2d(1)
        self.enc = nn.Sequential(ResBlock(1, 32, 2), ResBlock(32, 32, 2), ResBlock(32, d), ResBlock(d, d))
        self.spec, self.temp = Branch(d), Branch(d)
        self.drop = nn.Dropout(0.5)
        self.out = nn.Linear(6 * d, 1)

    def forward(self, wav):                          # wav (B, 64000) -> logits (B,)
        wav = wav.float()
        wav = (wav - wav.mean(1, keepdim=True)) / torch.sqrt(wav.var(1, keepdim=True) + 1e-7)
        h = self.ssl(wav).last_hidden_state          # (B, T, 1024)
        x = self.proj(h).transpose(1, 2).unsqueeze(1)  # (B, 1, 128, T)
        x = F.selu(self.bn0(F.max_pool2d(x, (3, 3))))
        x = self.enc(x)                              # (B, d, freq, time)
        s = x.abs().amax(dim=3).transpose(1, 2)      # spectral nodes (B, freq, d)
        t = x.abs().amax(dim=2).transpose(1, 2)      # temporal nodes (B, time, d)
        ms, ns = self.spec(s)
        mt, nt = self.temp(t)
        z = torch.cat([ms, ns.amax(1), ns.mean(1), mt, nt.amax(1), nt.mean(1)], dim=1)
        return self.out(self.drop(z)).squeeze(1)


@torch.no_grad()
def predict(model, loader, device, amp=True):
    model.eval()
    labels, scores = [], []
    for wav, y in loader:
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp and device.type == "cuda"):
            logits = model(wav.to(device))
        scores += torch.sigmoid(logits.float()).cpu().tolist()
        labels += y.tolist()
    return labels, scores


## Quick sanity check (1 minute)
Checks the model runs and prints the parameter count before the long training.

In [ ]:
import torch
from xlsr_aasist import XLSR_AASIST
m = XLSR_AASIST().cuda()
x = torch.randn(2, 64000).cuda()
with torch.autocast("cuda", dtype=torch.float16):
    print("logits:", m(x))
print("trainable params (M):", sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6)
del m, x; torch.cuda.empty_cache()

## Train
Settings are at the top of the cell. For an **ablation** (XLS-R without RawBoost) set `ALGO = 0` and `NAME = "xlsr_aasist_noaug"`.
If you get a CUDA out-of-memory error, set `BATCH = 8`. If loss becomes `nan`, set `USE_AMP = False`.

In [ ]:
import os, math, random, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LambdaLR
from xlsr_aasist import AudioDataset, XLSR_AASIST, predict, worker_init
from metrics import compute_metrics

NAME, ALGO, PROB = "xlsr_aasist_rb4", 4, 0.7      # same RawBoost setting as the CNN run
EPOCHS, BATCH, SEED, USE_AMP, WORKERS = 8, 16, 1, True, 2
LR_SSL, LR_BACK = 1e-5, 1e-4                       # small lr for XLS-R, bigger for the new back-end
OUT = "/content/drive/MyDrive/anti_spoof"; os.makedirs(OUT, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
train_dl = DataLoader(AudioDataset("data/train.csv", ALGO, PROB), batch_size=BATCH, shuffle=True, drop_last=True,
                      num_workers=WORKERS, worker_init_fn=worker_init, pin_memory=True, persistent_workers=True)
dev_dl = DataLoader(AudioDataset("data/dev.csv"), batch_size=BATCH * 2, num_workers=WORKERS)

model = XLSR_AASIST().to(device)
ssl_p = [p for p in model.ssl.parameters() if p.requires_grad]
ids = {id(p) for p in ssl_p}
back_p = [p for p in model.parameters() if p.requires_grad and id(p) not in ids]
opt = torch.optim.AdamW([{"params": ssl_p, "lr": LR_SSL}, {"params": back_p, "lr": LR_BACK}], weight_decay=1e-4)
total = EPOCHS * len(train_dl); warm = max(1, int(0.05 * total))
sched = LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
loss_fn = nn.BCEWithLogitsLoss()

best, ckpt = 100.0, f"{OUT}/{NAME}.pt"
for ep in range(1, EPOCHS + 1):
    model.train(); tot = 0.0
    for i, (wav, y) in enumerate(train_dl):
        wav, y = wav.to(device, non_blocking=True), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(wav)
        loss = loss_fn(logits.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item()
        if i % 50 == 0: print(f"  ep{ep} step {i}/{len(train_dl)} loss {loss.item():.4f}", flush=True)
    labels, scores = predict(model, dev_dl, device, USE_AMP)
    m = compute_metrics(labels, scores)
    print(f"Epoch {ep}: train_loss={tot/len(train_dl):.4f} | dev EER={m['EER']:.2f}% AUC={m['AUC']:.3f} F1={m['F1']:.3f}", flush=True)
    if m["EER"] < best:
        best = m["EER"]; torch.save(model.state_dict(), ckpt)
        print(f"  saved best model to Drive (dev EER {best:.2f}%)", flush=True)
print("Done. Best dev EER:", f"{best:.2f}%")

## Evaluate (same metrics.py as the CNN)
Loads the best checkpoint (picked by dev EER only) and tests on LA eval, MLAAD all, MLAAD matched and each MLAAD language. Results are appended to `results_colab.csv` on Drive.

In [ ]:
import os, pandas as pd, torch
from torch.utils.data import DataLoader
from xlsr_aasist import AudioDataset, XLSR_AASIST, predict
from metrics import compute_metrics

model = XLSR_AASIST().to(device)
model.load_state_dict(torch.load(ckpt, map_location=device)); model.eval()

sets = {"la_dev": "data/dev.csv", "la_eval": "data/eval.csv",
        "mlaad_all": "data/mlaad_all.csv", "mlaad_matched": "data/mlaad_matched.csv"}
ml = pd.read_csv("data/mlaad_all.csv")
os.makedirs("tmp", exist_ok=True)
for lang, g in ml.groupby("lang"):
    g.to_csv(f"tmp/mlaad_{lang}.csv", index=False); sets[f"mlaad_{lang}"] = f"tmp/mlaad_{lang}.csv"

rows = []
for sname, csv in sets.items():
    df = pd.read_csv(csv)
    labels, scores = predict(model, DataLoader(AudioDataset(csv), batch_size=32, num_workers=2), device, USE_AMP)
    m = compute_metrics(labels, scores)
    df["score"] = scores; df.to_csv(f"{OUT}/scores_{NAME}_{sname}.csv", index=False)   # per-file scores for later analysis
    rows.append({"name": NAME, "set": sname, "n": len(df), **m, "algo": ALGO, "prob": PROB, "epochs": EPOCHS, "seed": SEED})
    print(f"[{NAME} | {sname:14s}] n={len(df):5d}  EER={m['EER']:6.2f}%  AUC={m['AUC']:.3f}  F1={m['F1']:.3f}")
res = pd.DataFrame(rows)
f = f"{OUT}/results_colab.csv"
res.to_csv(f, mode="a", header=not os.path.exists(f), index=False)
print("saved ->", f)

## Optional: which TTS systems does the model miss?
Uses the saved per-file scores of the full MLAAD set (only systems with at least 5 clips).

In [ ]:
s = pd.read_csv(f"{OUT}/scores_{NAME}_mlaad_all.csv")
t = s.groupby("attack").score.agg(mean_score="mean", pct_pred_fake=lambda x: (x >= 0.5).mean() * 100, n="count")
print(t[t.n >= 5].sort_values("mean_score").round(2).to_string())